[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_08/MFM_ch8_sem_var/MFM_ch8_sem_var.ipynb)

# MFM_ch8_sem_var

Seminar 8, B1-B2 and B9: VaR backtests of six models on the S&P 500 with asymptotic and exact p-values and the DQ test; Basel traffic light and FRTB desk check for the GARCH-t VaR on the BET index; parametric bootstrap of the Kupiec test under estimation risk.

*Modelling Financial Markets (MFM), Chapter 8: Backtesting and Evaluating Risk Forecasts. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_8'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import sys
import subprocess
try:
    import arch  # GARCH estimation
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)
import pickle
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats, optimize
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')

## Data and backtesting tools

Market prices from the course repository (`data/market`); EUR/RON: BNR reference rate. Rolling one-day-ahead VaR/ES forecasts (1000-day window) and all backtests.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

# name -> (symbol, start date); indices/crypto: closing price
SOURCES = {
    'sp500':  ('GSPC.INDX', '1990-01-01'),
    'bet':    ('BET', '1997-09-19'),
    'btc':    ('BTC-USD.CC', '2014-09-17'),
    'eurron': ('REF:EUR', '2005-07-01'),        # BNR reference rate, since the introduction of the new leu
}
LABELS = {'sp500': 'S&P 500', 'bet': 'BET', 'btc': 'Bitcoin', 'eurron': 'EUR/RON'}
ASSETS = list(SOURCES)

_CACHE = {}


def read_market(symbol):
    """Read data/market/<SYMBOL>.csv locally or from the GitHub repository."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Official RON reference rate published by the BNR (yearly XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_price(name):
    """Price series (close) of the asset."""
    symbol, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    p = read_market(symbol)['close'].loc[start:]
    return p[p > 0].dropna()


def load_returns(name):
    """Daily log returns on the series' own calendar."""
    return np.log(load_price(name)).diff().dropna()


# =============================================================================
# VaR / ES FORECASTS ON A ROLLING WINDOW (one day ahead)
# =============================================================================
W = 1000                       # estimation window (observations)
REFIT = 20                     # GARCH / t / EVT parameters are re-estimated every 20 days
LEVELS = (0.01, 0.025, 0.05)   # tail probabilities: VaR 1%, VaR 2.5%, VaR 5%
A_ES = 0.025                   # ES 2.5% (FRTB)
K_TAIL = int(np.ceil(A_ES * W))   # number of tail order statistics (Z2 simulation)
MODELS = ['HS', 'Normal', 'Student-t', 'GARCH-t', 'FHS', 'GARCH-EVT']
EVAL_FROM = '2007-01-01'       # first possible evaluation day
EVT_Q = 0.90                   # POT threshold: 90% quantile of the standardised losses


def t_std_q(nu, a):
    """Quantile of order 1-a of the standardised Student-t distribution (unit variance)."""
    return stats.t.ppf(1 - a, nu) * np.sqrt((nu - 2) / nu)


def t_std_es(nu, a):
    """ES at level 1-a of the standardised Student-t distribution (unit variance)."""
    q = stats.t.ppf(1 - a, nu)
    return stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1) * np.sqrt((nu - 2) / nu)


def garch_fit(x, prev=None):
    """GARCH(1,1) with Student-t innovations by maximum likelihood; parameters in decimal units.
    The series is rescaled to unit variance for numerical stability; if the optimisation fails,
    the previous estimates are kept."""
    from arch import arch_model
    c = 1 / x.std()
    res = arch_model(c * x, mean='Constant', vol='GARCH', p=1, q=1, dist='t',
                     rescale=False).fit(disp='off', show_warning=False)
    mu, om, a, b, nu = res.params.values
    ok = res.convergence_flag == 0 and abs(mu) < 1 and a + b < 1 and om > 0
    if not ok and prev is not None:
        return prev
    return mu / c, om / c ** 2, a, b, nu


def garch_filter(x, mu, om, a, b, s2_0):
    """Conditional variance: s2[t] uses only x[:t]; the last element is the forecast."""
    e2 = (x - mu) ** 2
    s2 = np.empty(len(x) + 1)
    s2[0] = s2_0
    for t in range(len(x)):
        s2[t + 1] = om + a * e2[t] + b * s2[t]
    return s2


def gpd_tail(y, a_list, q=EVT_Q):
    """EVT (POT): GPD on the excesses of standardised losses over the q quantile; VaR/ES at levels 1-a."""
    u = np.quantile(y, q)
    exc = y[y > u] - u
    xi, _, beta = stats.genpareto.fit(exc, floc=0)
    pu = len(exc) / len(y)
    var = {a: u + beta / xi * ((a / pu) ** (-xi) - 1) for a in a_list}
    es = {a: (var[a] + beta - xi * u) / (1 - xi) for a in a_list}
    return dict(u=u, xi=xi, beta=beta, pu=pu, var=var, es=es)


def gpd_cdf(y, g, emp):
    """EVT distribution function: empirical below the threshold, GPD above it."""
    if y <= g['u']:
        return np.mean(emp <= y)
    base = 1 + g['xi'] * (y - g['u']) / g['beta']
    return 1.0 if base <= 0 else 1 - g['pu'] * base ** (-1 / g['xi'])


def rolling_forecasts(r, eval_from=EVAL_FROM, w=W, refit=REFIT):
    """One-day-ahead VaR (LEVELS) and ES (A_ES) forecasts for the loss L = -r, for all models.

    Result: dict model -> DataFrame (index = forecast day) with columns
    VaR1, VaR2.5, VaR5, ES2.5, ES1, pit (F(L_t)), scale, mu, plus information for tail simulation."""
    r = r.dropna()
    L = -r.values
    idx = r.index
    start = max(w, int(np.searchsorted(idx, pd.Timestamp(eval_from))))
    T = len(r) - start
    days = idx[start:]
    out = {m: {k: np.full(T, np.nan) for k in ['VaR1', 'VaR2.5', 'VaR5', 'ES2.5', 'ES1', 'pit', 'scale', 'mu', 'nu',
                                               'u', 'xi', 'beta', 'pu']} for m in MODELS}
    tails = {m: np.full((T, K_TAIL), np.nan) for m in ('HS', 'FHS')}
    lev = {0.01: 'VaR1', 0.025: 'VaR2.5', 0.05: 'VaR5'}
    Lcur = L[start:]

    # --- HS and Normal: vectorised over windows
    win = sliding_window_view(L[:-1], w)[start - w:]          # window for day t: L[t-w:t]
    srt = np.sort(win, axis=1)
    o = out['HS']
    for a, k in lev.items():
        o[k] = np.quantile(win, 1 - a, axis=1)
    for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
        v = np.quantile(win, 1 - a, axis=1)
        o[k] = np.nanmean(np.where(win >= v[:, None], win, np.nan), axis=1)
    o['pit'] = (np.sum(win < Lcur[:, None], axis=1) + 0.5 * np.sum(win == Lcur[:, None], axis=1)) / w
    o['scale'] = win.std(axis=1, ddof=1)
    o['mu'] = np.zeros(T)
    tails['HS'] = srt[:, -K_TAIL:]
    mu_w, sd_w = -win.mean(axis=1), win.std(axis=1, ddof=1)     # mean of returns, standard deviation
    o = out['Normal']
    for a, k in lev.items():
        o[k] = -mu_w + sd_w * stats.norm.ppf(1 - a)
    for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
        o[k] = -mu_w + sd_w * stats.norm.pdf(stats.norm.ppf(1 - a)) / a
    o['pit'] = stats.norm.cdf((Lcur + mu_w) / sd_w)
    o['scale'], o['mu'] = sd_w, mu_w

    # --- models re-estimated every `refit` days
    prev = None
    for b0 in range(0, T, refit):
        t0 = start + b0
        b1 = min(T, b0 + refit)
        x = r.values[t0 - w:t0]
        # unconditional Student-t: degrees of freedom, location and scale by MLE
        nu_t, loc_t, sc_t = stats.t.fit(x)
        nu_t = max(nu_t, 2.2)
        sd_t = sc_t * np.sqrt(nu_t / (nu_t - 2))                # implied standard deviation
        # GARCH(1,1)-t
        prev = garch_fit(x, prev if b0 > 0 else None)
        mu, om, al, be, nu = prev
        nu = max(nu, 2.2)
        xx = r.values[t0 - w:t0 + (b1 - b0)]
        s2 = garch_filter(xx, mu, om, al, be, np.var(x[:50]))
        z = (x - mu) / np.sqrt(s2[:w])
        y = np.sort(-z)                                         # standardised losses
        g = gpd_tail(y, (0.01, 0.025, 0.05))
        for t in range(b0, b1):
            j = w + (t - b0)
            sig = np.sqrt(s2[j])
            Lt = Lcur[t]
            # unconditional Student-t
            o = out['Student-t']
            for a, k in lev.items():
                o[k][t] = -loc_t + sd_t * t_std_q(nu_t, a)
            o['ES2.5'][t] = -loc_t + sd_t * t_std_es(nu_t, 0.025)
            o['ES1'][t] = -loc_t + sd_t * t_std_es(nu_t, 0.01)
            o['pit'][t] = stats.t.cdf((Lt + loc_t) / sc_t, nu_t)
            o['scale'][t], o['mu'][t], o['nu'][t] = sd_t, loc_t, nu_t
            # GARCH-t
            o = out['GARCH-t']
            for a, k in lev.items():
                o[k][t] = -mu + sig * t_std_q(nu, a)
            o['ES2.5'][t] = -mu + sig * t_std_es(nu, 0.025)
            o['ES1'][t] = -mu + sig * t_std_es(nu, 0.01)
            o['pit'][t] = stats.t.cdf((Lt + mu) / sig / np.sqrt((nu - 2) / nu), nu)
            o['scale'][t], o['mu'][t], o['nu'][t] = sig, mu, nu
            # FHS: empirical quantiles of the standardised losses
            o = out['FHS']
            for a, k in lev.items():
                o[k][t] = -mu + sig * np.quantile(y, 1 - a)
            for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
                q = np.quantile(y, 1 - a)
                o[k][t] = -mu + sig * y[y >= q].mean()
            o['pit'][t] = np.mean(y < (Lt + mu) / sig)
            o['scale'][t], o['mu'][t] = sig, mu
            tails['FHS'][t] = -mu + sig * y[-K_TAIL:]
            # GARCH-EVT
            o = out['GARCH-EVT']
            for a, k in lev.items():
                o[k][t] = -mu + sig * g['var'][a]
            o['ES2.5'][t] = -mu + sig * g['es'][0.025]
            o['ES1'][t] = -mu + sig * g['es'][0.01]
            o['pit'][t] = gpd_cdf((Lt + mu) / sig, g, y)
            o['scale'][t], o['mu'][t] = sig, mu
            o['u'][t], o['xi'][t], o['beta'][t], o['pu'][t] = g['u'], g['xi'], g['beta'], g['pu']
    res = {}
    for m in MODELS:
        df = pd.DataFrame(out[m], index=days)
        df['L'] = Lcur
        res[m] = df
    return res, tails


def get_forecasts(name):
    """Forecasts for one asset, with a temporary cache (the computation takes a few minutes)."""
    path = os.path.join(tempfile.gettempdir(), f'mfm_ch8_fc_{name}_{W}_{REFIT}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    fc = rolling_forecasts(load_returns(name))
    with open(path, 'wb') as f:
        pickle.dump(fc, f)
    return fc


# =============================================================================
# VaR TESTS
# =============================================================================
def kupiec(hits, p):
    """Kupiec POF test: LR_uc ~ chi2(1)."""
    hits = np.asarray(hits, int)
    T, x = len(hits), hits.sum()
    ph = x / T
    ll0 = (T - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (T - x) * np.log(1 - ph) + x * np.log(ph) if 0 < x < T else 0.0
    lr = -2 * (ll0 - ll1)
    return dict(T=T, x=int(x), rate=ph, LR=lr, p=stats.chi2.sf(lr, 1))


def transitions(hits):
    """Counts of the transitions n00, n01, n10, n11 of the hit sequence."""
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    return dict(n00=int(np.sum((a == 0) & (b == 0))), n01=int(np.sum((a == 0) & (b == 1))),
                n10=int(np.sum((a == 1) & (b == 0))), n11=int(np.sum((a == 1) & (b == 1))))


def christoffersen_from_counts(n00, n01, n10, n11, p):
    """LR_ind (independence, first-order Markov chain) and LR_cc = LR_uc + LR_ind."""
    def xlogy(n, q):
        return n * np.log(q) if n > 0 else 0.0
    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l_ind = xlogy(n00, 1 - pi01) + xlogy(n01, pi01) + xlogy(n10, 1 - pi11) + xlogy(n11, pi11)
    l_0 = xlogy(n00 + n10, 1 - pi) + xlogy(n01 + n11, pi)
    lr_ind = -2 * (l_0 - l_ind)
    T, x = n00 + n01 + n10 + n11, n01 + n11
    lr_uc = -2 * (xlogy(T - x, 1 - p) + xlogy(x, p) - xlogy(T - x, 1 - pi) - xlogy(x, pi))
    lr_cc = lr_uc + lr_ind
    return dict(pi01=pi01, pi11=pi11, pi=pi, LR_ind=lr_ind, p_ind=stats.chi2.sf(lr_ind, 1),
                LR_uc=lr_uc, LR_cc=lr_cc, p_cc=stats.chi2.sf(lr_cc, 2))


def christoffersen(hits, p):
    return christoffersen_from_counts(**transitions(hits), p=p)


def durations(hits):
    """Durations (days) between breaches, with censoring flags for the first and last duration."""
    h = np.asarray(hits, int)
    pos = np.flatnonzero(h)
    if len(pos) == 0:
        return np.array([len(h)]), 1, 1
    D = np.diff(np.r_[-1, pos, len(h) - 1 if h[-1] == 0 else pos[-1]])
    D = D[D > 0]
    c1 = 1 if h[0] == 0 else 0
    cn = 1 if h[-1] == 0 else 0
    return D.astype(float), c1, cn


def duration_test(hits):
    """Christoffersen-Pelletier (2004) duration test: continuous Weibull approximation, H0: b = 1 (memoryless)."""
    D, c1, cn = durations(hits)
    if len(D) < 3:
        return dict(b=np.nan, LR=np.nan, p=np.nan, n=len(D))

    def nll(par, fix_b=False):
        a = np.exp(par[0])
        b = 1.0 if fix_b else np.exp(par[1])
        lf = np.log(b) + b * np.log(a) + (b - 1) * np.log(D) - (a * D) ** b
        ls = -(a * D) ** b
        ll = lf.copy()
        if c1:
            ll[0] = ls[0]
        if cn:
            ll[-1] = ls[-1]
        return -ll.sum()
    a0 = np.log(1 / D.mean())
    r1 = optimize.minimize(nll, [a0, 0.0], method='Nelder-Mead', options=dict(xatol=1e-8, fatol=1e-10, maxiter=4000))
    r0 = optimize.minimize(lambda p: nll(p, True), [a0], method='Nelder-Mead')
    lr = max(2 * (r0.fun - r1.fun), 0.0)
    return dict(b=float(np.exp(r1.x[1])), LR=lr, p=stats.chi2.sf(lr, 1), n=len(D))


def traffic_light(x, T=250, p=0.01):
    """Basel zone (green/yellow/red) for x exceptions in T days; cumulative Binomial probability."""
    cum = stats.binom.cdf(x, T, p)
    zone = 'green' if cum < 0.95 else ('yellow' if cum < 0.9999 else 'red')
    return zone, cum


# =============================================================================
# ES TESTS
# =============================================================================
def mcneil_frey(L, var, es, scale, B=5000, seed=0):
    """Exceedance residuals (L - ES)/sigma on days with L > VaR; H0: mean 0, H1: mean > 0 (i.i.d. bootstrap)."""
    hit = L > var
    e = ((L - es) / scale)[hit]
    n = len(e)
    if n < 3:
        return dict(n=n, mean=np.nan, t=np.nan, p=np.nan)
    tstat = e.mean() / (e.std(ddof=1) / np.sqrt(n))
    rng = np.random.default_rng(seed)
    ec = e - e.mean()
    bs = rng.choice(ec, (B, n), replace=True)
    tb = bs.mean(1) / (bs.std(1, ddof=1) / np.sqrt(n))
    return dict(n=n, mean=e.mean(), t=tstat, p=float(np.mean(tb >= tstat)))


def acerbi_szekely(L, var, es, a=A_ES):
    """Acerbi-Szekely (2014) Z1 and Z2 statistics, with losses positive."""
    hit = L > var
    T, n = len(L), hit.sum()
    z1 = 1 - np.mean(L[hit] / es[hit]) if n > 0 else np.nan
    z2 = 1 - np.sum(L[hit] / es[hit]) / (T * a)
    return z1, z2


def tail_sampler(df, model, tails, U):
    """Inverse loss distribution function for levels U > 1 - A_ES (matrix T x M)."""
    mu, sc = df['mu'].values[:, None], df['scale'].values[:, None]
    if model in ('HS', 'FHS'):
        k = np.clip(np.ceil((U - (1 - A_ES)) / A_ES * K_TAIL).astype(int) - 1, 0, K_TAIL - 1)
        return np.take_along_axis(tails[model], k, axis=1)
    if model == 'Normal':
        return -mu + sc * stats.norm.ppf(U)
    if model in ('Student-t', 'GARCH-t'):
        nu = df['nu'].values[:, None]
        return -mu + sc * stats.t.ppf(U, nu) * np.sqrt((nu - 2) / nu)
    if model == 'GARCH-EVT':
        u, xi, be, pu = (df[c].values[:, None] for c in ('u', 'xi', 'beta', 'pu'))
        return -mu + sc * (u + be / xi * (((1 - U) / pu) ** (-xi) - 1))
    raise ValueError(model)


def z_pvalues(df, model, tails, M=2000, seed=1, a=A_ES):
    """Simulated p-values under H0 (the model's forecast distribution) for Z1 and Z2."""
    L, var, es = df['L'].values, df['VaR2.5'].values, df['ES2.5'].values
    z1, z2 = acerbi_szekely(L, var, es, a)
    rng = np.random.default_rng(seed)
    T = len(L)
    z1s, z2s = np.empty(M), np.empty(M)
    step = 250
    for m0 in range(0, M, step):
        k = min(step, M - m0)
        U = rng.uniform(size=(T, k))
        hit = U > 1 - a
        Us = np.where(hit, U, 1 - a / 2)
        Ls = tail_sampler(df, model, tails, Us)
        ratio = np.where(hit, Ls / es[:, None], 0.0)
        z2s[m0:m0 + k] = 1 - ratio.sum(0) / (T * a)
        nh = hit.sum(0)
        z1s[m0:m0 + k] = 1 - ratio.sum(0) / np.maximum(nh, 1)
    return dict(Z1=z1, Z2=z2, p1=float(np.mean(z1s <= z1)), p2=float(np.mean(z2s <= z2)),
                crit2=float(np.quantile(z2s, 0.05)))


def du_escanciano(pit, a=A_ES, lags=5):
    """Du-Escanciano (2017): cumulative violation H_t = (1/a)(u_t - (1-a)) 1{u_t > 1-a};
    unconditional test U_ES ~ N(0,1) and conditional test C_ES (Box-Pierce, chi2(lags))."""
    u = np.clip(np.asarray(pit), 0, 1)
    H = (u - (1 - a)) / a * (u > 1 - a)
    T = len(H)
    U = np.sqrt(T) * (H.mean() - a / 2) / np.sqrt(a * (1 / 3 - a / 4))
    hc = H - a / 2
    g0 = np.mean(hc ** 2)
    rho = np.array([np.mean(hc[j:] * hc[:-j]) / g0 for j in range(1, lags + 1)])
    C = T * np.sum(rho ** 2)
    return dict(meanH=H.mean(), U=U, pU=2 * stats.norm.sf(abs(U)), C=C, pC=stats.chi2.sf(C, lags))


# =============================================================================
# FZ0 LOSS, DIEBOLD-MARIANO, MCS
# =============================================================================
def fz0(L, var, es, a=A_ES):
    """FZ0 loss (Patton, Ziegel & Chen, 2019), written for positive losses:
    FZ0 = (1/(a ES)) 1{L > VaR} (L - VaR) + VaR/ES + log(ES) - 1  (ES > 0)."""
    return (L > var) * (L - var) / (a * es) + var / es + np.log(es) - 1


def pinball(L, var, a):
    """Quantile (tick) loss: (1{L > VaR} - a)(L - VaR) >= 0, minimised in expectation by the 1-a quantile."""
    return ((L > var).astype(float) - a) * (L - var)


def nw_var(d, lags=None):
    """Long-run variance (Newey-West, Bartlett kernel)."""
    d = np.asarray(d) - np.mean(d)
    T = len(d)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    v = np.mean(d ** 2)
    for j in range(1, lags + 1):
        v += 2 * (1 - j / (lags + 1)) * np.dot(d[j:], d[:-j]) / T     # autocovariance with divisor T (PSD)
    return v


def diebold_mariano(la, lb, lags=None):
    """DM = mean(d) / sqrt(LRV/T), d = la - lb; negative => A has the lower average loss."""
    d = np.asarray(la) - np.asarray(lb)
    dm = d.mean() / np.sqrt(nw_var(d, lags) / len(d))
    return dict(dbar=d.mean(), DM=dm, p=2 * stats.norm.sf(abs(dm)))


def mcs(losses, alpha=0.10, B=1000, block=10, seed=2):
    """Model confidence set (Hansen, Lunde & Nason, 2011), T_max statistic,
    circular block bootstrap. losses: DataFrame T x m. Result: MCS p-values."""
    rng = np.random.default_rng(seed)
    X = losses.values
    T, m = X.shape
    nb = int(np.ceil(T / block))
    idx = (rng.integers(0, T, (B, nb))[:, :, None] + np.arange(block)) % T
    idx = idx.reshape(B, -1)[:, :T]
    Lbar = X.mean(0)
    Lb = np.stack([X[i].mean(0) for i in idx])           # B x m
    alive = list(range(m))
    pvals, p_run = {}, 0.0
    while len(alive) > 1:
        a = np.array(alive)
        d = Lbar[a] - Lbar[a].mean()
        db = Lb[:, a] - Lb[:, a].mean(1, keepdims=True)
        se = np.sqrt(np.mean((db - d) ** 2, axis=0))
        t = d / se
        tb = ((db - d) / se).max(1)
        p = float(np.mean(tb >= t.max()))
        p_run = max(p_run, p)
        worst = a[np.argmax(t)]
        pvals[losses.columns[worst]] = p_run
        alive.remove(worst)
    pvals[losses.columns[alive[0]]] = 1.0
    return pd.Series(pvals)[losses.columns]


# =============================================================================
# CONFORMAL PREDICTION FOR VaR
# =============================================================================
def conformal_var(r, a=0.01, n_cal=250, n_sd=250, gamma=0.005, eval_from=EVAL_FROM):
    """Rolling split conformal VaR and ACI (Gibbs & Candes, 2021).

    Nonconformity score: s_t = L_t / sd_t, sd_t = standard deviation of the previous n_sd returns.
    VaR_t = sd_t * the order statistic k = ceil((n+1)(1-a)) of the last n_cal scores (+inf if k > n).
    ACI: a_{t+1} = a_t + gamma (a - err_t); if the required rank exceeds n, VaR = +inf.
    Rolling-standardised scores are only approximately exchangeable, even for i.i.d. returns."""
    r = r.dropna()
    L = -r.values
    sd = pd.Series(r.values).rolling(n_sd).std().shift(1).values   # only information up to t-1
    s = L / sd
    start = max(n_sd + n_cal + 1, int(np.searchsorted(r.index, pd.Timestamp(eval_from))))
    T = len(r) - start
    var_s, var_a, a_path = np.empty(T), np.empty(T), np.empty(T)
    at = a
    for i in range(T):
        t = start + i
        cal = np.sort(s[t - n_cal:t])
        k = int(np.ceil((n_cal + 1) * (1 - a)))
        var_s[i] = np.inf if k > n_cal else sd[t] * cal[k - 1]
        ka = int(np.ceil((n_cal + 1) * (1 - at)))
        var_a[i] = np.inf if ka > n_cal else sd[t] * cal[max(ka, 1) - 1]
        a_path[i] = at
        err = float(L[t] > var_a[i])
        at = at + gamma * (a - err)
    return pd.DataFrame({'L': L[start:], 'VaR_split': var_s, 'VaR_aci': var_a, 'alpha_t': a_path,
                         'sd': sd[start:]}, index=r.index[start:])


def regimes(r, idx, n=20, q=0.90, w=W, min_obs=250):
    """Crisis days: volatility over the n days before day t (excluding day t) above the q quantile of the
    same volatilities over the last w days (the models' estimation window, at least min_obs values);
    the indicator and its threshold use returns up to t-1 only, so they are known ex ante."""
    rv = r.rolling(n).std().shift(1)
    cut = rv.rolling(w, min_periods=min_obs).quantile(q)
    return (rv > cut).reindex(idx).fillna(False).astype(bool)

## Style and helpers

In [ ]:
# MFM standard style (as in SFM): transparent + English labels + legend at the bottom
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Brand colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal = '#17A2B8'
MCOL = {'HS': Teal, 'Normal': Amber, 'Student-t': Purple, 'GARCH-t': MainBlue, 'FHS': Forest, 'GARCH-EVT': IDAred}
PV_CMAP = LinearSegmentedColormap.from_list('pv', [IDAred, '#F4B6B6', '#FFFFFF', '#BFD8BF', Forest])


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


_FC = {}


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def pv_heatmap(ax, P, fmt='{:.2f}', title=None):
    """Heat map of p-values: red = rejection, green = no rejection (log scale)."""
    Z = np.log10(np.clip(P.values.astype(float), 1e-4, 1))
    ax.imshow(Z, cmap=PV_CMAP, vmin=-4, vmax=0, aspect='auto')
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            v = P.values[i, j]
            txt = '<0.001' if v < 0.001 else (f'{v:.3f}' if v < 0.01 else fmt.format(v))
            ax.text(j, i, txt, ha='center', va='center', fontsize=7)
    ax.set_xticks(range(P.shape[1]), P.columns, fontsize=7.5)
    ax.set_yticks(range(P.shape[0]), P.index, fontsize=7.5)
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=9)


def fc(name):
    if name not in _FC:
        _FC[name] = get_forecasts(name)
    return _FC[name]


def hits(name, model, col='VaR1'):
    df = fc(name)[0][model]
    return (df['L'] > df[col]).astype(int)

## Analysis

In [ ]:
def rolling_exceptions(name, model, col='VaR1', n=250):
    return hits(name, model, col).rolling(n).sum().dropna()


def b1_sp500_var():
    """VaR 1% tests on the S&P 500; asymptotic versus exact (Binomial) p-value."""
    out = {}
    for m in MODELS:
        h = hits('sp500', m).values
        k, c, d = kupiec(h, 0.01), christoffersen(h, 0.01), duration_test(h)
        exact = 2 * min(stats.binom.cdf(k['x'], k['T'], 0.01), stats.binom.sf(k['x'] - 1, k['T'], 0.01))
        out[m] = dict(x=k['x'], T=k['T'], rate=k['rate'], p_uc=k['p'], p_exact=float(min(exact, 1)),
                      p_ind=c['p_ind'], p_cc=c['p_cc'], b=d['b'], p_dur=d['p'])
    # small sample: one year (2008) for GARCH-t
    h = hits('sp500', 'GARCH-t').loc['2008'].values
    k = kupiec(h, 0.01)
    out['small'] = dict(T=k['T'], x=k['x'], p_asy=k['p'],
                        p_exact=float(min(1, 2 * min(stats.binom.cdf(k['x'], k['T'], 0.01),
                                                     stats.binom.sf(k['x'] - 1, k['T'], 0.01)))))
    return out


def b2_traffic_bet():
    """Basel traffic light for the GARCH-t VaR 1% on the BET: last 250 days and share of time in each zone."""
    x = rolling_exceptions('bet', 'GARCH-t')
    last = int(x.iloc[-1])
    zone, cum = traffic_light(last)
    mult = {0: 1.5, 1: 1.5, 2: 1.5, 3: 1.5, 4: 1.5, 5: 1.7, 6: 1.76, 7: 1.83, 8: 1.88, 9: 1.92}.get(last, 2.0)
    fig, ax = plt.subplots(figsize=(6.8, 2.9))
    ax.axhspan(-0.5, 4.5, color=Forest, alpha=0.10, lw=0)
    ax.axhspan(4.5, 9.5, color='#F1C40F', alpha=0.15, lw=0)
    ax.axhspan(9.5, 30, color=IDAred, alpha=0.10, lw=0)
    for m in ('HS', 'GARCH-t', 'GARCH-EVT'):
        xm = rolling_exceptions('bet', m)
        ax.plot(xm.index, xm, color=MCOL[m], lw=0.9, label=m)
    ax.set_ylim(-0.5, 26)
    ax.set_ylabel('Exceptions of VaR 1%\nin the last 250 days (BET)')
    legend_outside_bottom(ax, ncol=3, y=-0.13)
    save_fig('ch8_sem_traffic_bet')
    h = hits('bet', 'GARCH-t')
    return dict(last=last, zone=zone, cum=cum, mult=mult, window=[str(h.index[-250].date()), str(h.index[-1].date())],
                share_green=float(np.mean(x <= 4)), share_yellow=float(np.mean((x >= 5) & (x <= 9))),
                share_red=float(np.mean(x >= 10)), max=int(x.max()), max_day=str(x.idxmax().date()),
                x975=int(hits('bet', 'GARCH-t', 'VaR2.5').iloc[-250:].sum()))


def dq_test(hit, var, a, lags=4):
    """Dynamic Quantile test (Engle & Manganelli, 2004): Hit_t = I_t - a regressed on
    X_t = (1, Hit_{t-1..t-lags}, VaR_t); DQ = b'X'Xb / (a(1-a)) ~ chi2(k)."""
    h = np.asarray(hit, float) - a
    v = np.asarray(var, float)
    y = h[lags:]
    X = np.column_stack([np.ones(len(y))] + [h[lags - j:len(h) - j] for j in range(1, lags + 1)] + [v[lags:]])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    dq = float(b @ X.T @ X @ b / (a * (1 - a)))
    return dict(DQ=dq, df=X.shape[1], p=float(stats.chi2.sf(dq, X.shape[1])))


def b1_dq_sp500():
    """DQ test (Engle & Manganelli, 2004) for the six VaR 1% models on the S&P 500."""
    F = fc('sp500')[0]
    return {m: dq_test((F[m]['L'] > F[m]['VaR1']).astype(int).values, F[m]['VaR1'].values, 0.01) for m in MODELS}


def sim_garch_t(par, n, reps, rng, burn=500):
    """GARCH(1,1)-t paths (standardised t innovations); returns the returns and sigma_t."""
    mu, om, a, b, nu = par
    z = rng.standard_t(nu, size=(reps, n + burn)) * np.sqrt((nu - 2) / nu)
    r = np.empty((reps, n + burn))
    s2 = np.full(reps, om / (1 - a - b))
    S = np.empty((reps, n + burn))
    for t in range(n + burn):
        S[:, t] = np.sqrt(s2)
        r[:, t] = mu + S[:, t] * z[:, t]
        s2 = om + a * (r[:, t] - mu) ** 2 + b * s2
    return r[:, burn:], S[:, burn:]


def mc_size_power(reps=1000, seed=8, Ts=(250, 1000, 5000), a=0.01, w=1000):
    """Monte Carlo: GARCH(1,1)-t DGP with the parameters estimated on the S&P 500, 1990-2026.
    VaR 1% forecasts: correct model (true parameters), Normal-GARCH (true sigma, Normal tail), HS (1000 days).
    Rejection rate at 5% for POF, CC, durations (Weibull) and DQ; undefined p (too few breaches) = no rejection."""
    r = load_returns('sp500').values
    par = garch_fit(r)
    mu, om, al, be, nu = par
    rng = np.random.default_rng(seed)
    Tmax = max(Ts)
    R, S = sim_garch_t(par, Tmax + w, reps, rng)
    L = -R[:, w:]
    q_t, q_n = t_std_q(nu, a), stats.norm.ppf(1 - a)
    out = {}
    for m in ('Correct', 'Normal-GARCH', 'HS'):
        rej = {T: {k: 0 for k in ('POF', 'CC', 'DUR', 'DQ')} for T in Ts}
        nan_dur = {T: 0 for T in Ts}
        for i in range(reps):
            if m == 'Correct':
                v = -mu + S[i, w:] * q_t
            elif m == 'Normal-GARCH':
                v = -mu + S[i, w:] * q_n
            else:
                v = pd.Series(-R[i]).rolling(w).quantile(1 - a).shift(1).values[w:]
            I = (L[i] > v).astype(int)
            for T in Ts:
                h, vv = I[:T], v[:T]
                rej[T]['POF'] += kupiec(h, a)['p'] < 0.05
                rej[T]['CC'] += christoffersen(h, a)['p_cc'] < 0.05 if h.sum() > 0 else 0
                d = duration_test(h)['p']
                nan_dur[T] += np.isnan(d)
                rej[T]['DUR'] += (d < 0.05) if not np.isnan(d) else 0
                rej[T]['DQ'] += dq_test(h, vv, a)['p'] < 0.05
        out[m] = {str(T): dict({k: v / reps for k, v in rej[T].items()}, dur_nan=nan_dur[T] / reps) for T in Ts}
    out['par'] = dict(mu=mu, omega=om, alpha=al, beta=be, nu=nu)
    out['reps'] = reps
    return out


def mc_estimation_risk(reps=400, seed=9, Rw=1000, Ps=(250, 1000, 5000), a=0.01):
    """Estimation risk: GARCH-t estimated once on R = 1000 days (fixed scheme), VaR 1% over P days.
    POF rejection rate at 5% with estimated versus true parameters, on the same paths."""
    r0 = load_returns('sp500').values
    par = garch_fit(r0)
    rng = np.random.default_rng(seed)
    Pm = max(Ps)
    R, S = sim_garch_t(par, Rw + Pm, reps, rng)
    mu0, _, _, _, nu0 = par
    rej_est = {P: 0 for P in Ps}
    rej_true = {P: 0 for P in Ps}
    rate_est = {P: [] for P in Ps}
    for i in range(reps):
        x = R[i, :Rw]
        mu, om, al, be, nu = garch_fit(x)
        nu = max(nu, 2.2)
        s2 = garch_filter(R[i], mu, om, al, be, np.var(x[:50]))
        v_est = -mu + np.sqrt(s2[Rw:Rw + Pm]) * t_std_q(nu, a)
        v_true = -mu0 + S[i, Rw:] * t_std_q(nu0, a)
        L = -R[i, Rw:]
        for P in Ps:
            he = (L[:P] > v_est[:P]).astype(int)
            ht = (L[:P] > v_true[:P]).astype(int)
            rej_est[P] += kupiec(he, a)['p'] < 0.05
            rej_true[P] += kupiec(ht, a)['p'] < 0.05
            rate_est[P].append(he.mean())
    return {str(P): dict(est=rej_est[P] / reps, true=rej_true[P] / reps,
                         sd_rate=float(np.std(rate_est[P])), sd_binom=float(np.sqrt(a * (1 - a) / P)))
            for P in Ps} | dict(reps=reps)


def b9_estimation_risk(reps=400):
    """Parametric bootstrap under estimation risk: GARCH-t re-estimated on every simulated path."""
    return mc_estimation_risk(reps=reps)


def _bars_legend(fig, axs, ncol=4, y=0.0):
    hs, ls = [], []
    for ax in np.atleast_1d(axs):
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in ls:
                hs.append(hh)
                ls.append(ll)
    fig.legend(hs, ls, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def ch_b1(b1=None, dq=None, MC=None):
    """B1: ten dated days of GARCH-t; breach raster and duration survival; p-value heat map and simulated size."""
    F = fc('sp500')[0]
    gt = F['GARCH-t']
    h = (gt['L'] > gt['VaR1']).astype(int)
    h8 = h.loc['2008']
    roll = h8.rolling(10).sum()
    end = roll.idxmax()
    i = h8.index.get_loc(end)
    w = gt.loc[h8.index[i - 9]:end]
    ten = [dict(date=str(d.date()), L=float(100 * r['L']), VaR=float(100 * r['VaR1']), hit=int(r['L'] > r['VaR1']))
           for d, r in w.iterrows()]
    b1 = b1 or b1_sp500_var()
    dq = dq or b1_dq_sp500()
    MC = MC or mc_size_power()
    # timing chart
    fig, axs = plt.subplots(1, 2, figsize=(7.6, 2.8), gridspec_kw=dict(width_ratios=[1.6, 1]))
    ax = axs[0]
    for k, m in enumerate(MODELS):
        hm = (F[m]['L'] > F[m]['VaR1'])
        ax.vlines(F[m].index[hm], k - 0.38, k + 0.38, color=MCOL[m], lw=0.7, label=m)
    ax.set_yticks(range(6), [f'{m} ({b1[m]["x"]})' for m in MODELS], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlabel('Breach days of VaR 1%, S&P 500 (count in brackets)')
    ax = axs[1]
    for m in MODELS:
        D, _, _ = durations((F[m]['L'] > F[m]['VaR1']).astype(int).values)
        Ds = np.sort(D)
        ax.step(Ds, 1 - np.arange(1, len(Ds) + 1) / (len(Ds) + 1), where='post', color=MCOL[m], lw=1.0)
    d = np.linspace(1, 900, 200)
    ax.plot(d, 0.99 ** d, color='black', ls='--', lw=0.9, label='No memory (geometric, 1%)')
    ax.set_yscale('log')
    ax.set_xlabel('Days between breaches d')
    ax.set_ylabel('Share of durations > d')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=7)
    save_fig('ch8_sem_b1_timing')
    # results chart
    P = pd.DataFrame({'Kupiec': [b1[m]['p_uc'] for m in MODELS], 'exact': [b1[m]['p_exact'] for m in MODELS],
                      'indep.': [b1[m]['p_ind'] for m in MODELS], 'CC': [b1[m]['p_cc'] for m in MODELS],
                      'duration': [b1[m]['p_dur'] for m in MODELS], 'DQ': [dq[m]['p'] for m in MODELS]},
                     index=MODELS)
    fig, axs = plt.subplots(1, 2, figsize=(7.6, 2.8), gridspec_kw=dict(width_ratios=[1.35, 1]))
    pv_heatmap(axs[0], P, title='p-values, VaR 1%, S&P 500 (log colour scale)')
    ax = axs[1]
    tests = ('POF', 'CC', 'DUR', 'DQ')
    reps = MC['reps']
    for j, (T, col) in enumerate((('250', MainBlue), ('5000', IDAred))):
        v = np.array([MC['Correct'][T][t] for t in tests])
        se = np.sqrt(v * (1 - v) / reps)
        ax.bar(np.arange(4) + (j - 0.5) * 0.36, 100 * v, 0.36, color=col, yerr=196 * se, capsize=2,
               error_kw=dict(lw=0.7), label=f'Simulated size, T = {T}')
    ax.axhline(5, color='black', ls='--', lw=0.8, label='Nominal 5%')
    ax.set_xticks(range(4), ['Kupiec', 'CC', 'duration', 'DQ'])
    ax.set_ylabel('Rejection rate of a correct model (%)')
    ax.set_title('Size of the battery (bars: ±1.96 MC s.e.)', fontsize=9)
    plt.tight_layout()
    _bars_legend(fig, axs[1:], ncol=3)
    save_fig('ch8_sem_b1_results')
    return dict(ten=ten, x10=int(sum(t['hit'] for t in ten)))


def ch_b9(b9=None):
    """B9: Kupiec rejection rates of a correct GARCH-t model, estimated versus true parameters, against P/R."""
    b9 = b9 or b9_estimation_risk()
    reps = b9['reps']
    Ps = ('250', '1000', '5000')
    x = np.array([int(P) / 1000 for P in Ps])
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    for key, col, lab, mk in (('est', IDAred, 'Estimated parameters (R = 1000)', 'o'),
                              ('true', MainBlue, 'True parameters', 's')):
        v = np.array([b9[P][key] for P in Ps])
        se = np.sqrt(v * (1 - v) / reps)
        ax.errorbar(x, 100 * v, yerr=196 * se, fmt=mk + '-', color=col, capsize=3, ms=4, lw=1.1, label=lab)
    ax.axhline(5, color='black', ls='--', lw=0.8, label='Nominal 5%')
    ax.set_xscale('log')
    ax.set_xticks(x, ['0.25', '1', '5'])
    ax.set_xlabel('P / R (out-of-sample days / estimation days)')
    ax.set_ylabel('Kupiec rejection rate at 5% (%)')
    legend_outside_bottom(ax, ncol=3, y=-0.2)
    save_fig('ch8_sem_b9_estrisk')
    return {P: dict(se_est=float(np.sqrt(b9[P]['est'] * (1 - b9[P]['est']) / reps)),
                    se_true=float(np.sqrt(b9[P]['true'] * (1 - b9[P]['true']) / reps))) for P in Ps}


def c2_fixture(n=400, day=320, shock=-0.08, seed=8):
    """C2 price fixture: 400 prices from 100, daily log returns 1% x standardised Student-t4, seed 8,
    with a return of -8% on day 320."""
    rng = np.random.default_rng(seed)
    r = 0.01 * rng.standard_t(4, n) / np.sqrt(2)
    r[day] = shock
    P = 100 * np.exp(np.r_[0, np.cumsum(r)])
    return P


def ch_c2(W=250, p=0.01):
    """C2: VaR from the wrong window r[t-W:t+1] (includes the day forecast) versus the lagged window r[t-W:t]."""
    P = c2_fixture()
    r = np.diff(np.log(P))
    idx = np.arange(W, len(r))
    v_bad = np.array([-np.quantile(r[t - W:t + 1], p) for t in idx])
    v_ok = np.array([-np.quantile(r[t - W:t], p) for t in idx])
    L = -r[W:]
    I_bad, I_ok = (L > v_bad).astype(int), (L > v_ok).astype(int)
    fig, ax = plt.subplots(figsize=(6.8, 2.8))
    sel = slice(300 - W, 345 - W)
    days = idx[sel]
    ax.bar(days, 100 * L[sel], color=Orange, alpha=0.6, width=0.8, label='Daily loss (fixture)')
    ax.step(days, 100 * v_bad[sel], where='mid', color=IDAred, lw=1.2, label='VaR from r[t−W : t+1] (includes day t)')
    ax.step(days, 100 * v_ok[sel], where='mid', color=MainBlue, lw=1.2, ls='--', label='VaR from r[t−W : t] (lagged)')
    ax.annotate('−8% on day 320: the wrong VaR already\nuses it (3.84%); the lagged VaR (2.28%)\nmoves only on day 321', (320.4, 3.84), xytext=(326, 5.6),
                fontsize=7.5, color='black', arrowprops=dict(arrowstyle='->', lw=0.7, color='black'))
    ax.set_xlabel('Day of the fixture')
    ax.set_ylabel('Loss / VaR (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch8_sem_c2_lookahead')
    rs = load_returns('sp500')
    rr = rs.values
    jj = np.arange(W, len(rr))
    vb = np.array([-np.quantile(rr[t - W:t + 1], p) for t in jj])
    vo = np.array([-np.quantile(rr[t - W:t], p) for t in jj])
    Ls = -rr[W:]
    return dict(x_bad=int(I_bad.sum()), x_ok=int(I_ok.sum()), T=len(L), sp_bad=int((Ls > vb).sum()),
                sp_ok=int((Ls > vo).sum()), sp_T=len(Ls), sp_from=str(rs.index[W].date()), var_bad=float(100 * v_bad[320 - W]),
                var_ok=float(100 * v_ok[320 - W]), var_ok_after=float(100 * v_ok[321 - W]),
                P0=float(P[0]), P320=float(P[320]), P321=float(P[321]))

## Run

In [ ]:
b1 = b1_sp500_var()
dq = b1_dq_sp500()
print(b1, dq)
print(b2_traffic_bet())
b9 = b9_estimation_risk()
print(b9)
print(ch_b1(b1, dq))
print(ch_b9(b9))
print(ch_c2())

![ch8_sem_traffic_bet](./ch8_sem_traffic_bet.png)

Output: `ch8_sem_traffic_bet.pdf`

![ch8_sem_b1_timing](./ch8_sem_b1_timing.png)

Output: `ch8_sem_b1_timing.pdf`

![ch8_sem_b1_results](./ch8_sem_b1_results.png)

Output: `ch8_sem_b1_results.pdf`

![ch8_sem_b9_estrisk](./ch8_sem_b9_estrisk.png)

Output: `ch8_sem_b9_estrisk.pdf`

![ch8_sem_c2_lookahead](./ch8_sem_c2_lookahead.png)

Output: `ch8_sem_c2_lookahead.pdf`